# Baseline gsplat Training (Backend-Equivalent)

This notebook runs the same gsplat training engine used by your backend,
without needing the API server/runtime orchestration.


In [15]:
from pathlib import Path
import json
import logging
import os
import sys

# Ensure repository root is importable when running from notebooks/.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'bimba3d_backend').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

# Paths
IMAGE_DIR = Path(r'E:\Thesis\tests\b5740a69-5358-4702-a6d6-1bc57147e0a1\images_resized')
COLMAP_DIR = Path(r'E:\Thesis\tests\b5740a69-5358-4702-a6d6-1bc57147e0a1\outputs\sparse')
OUTPUT_DIR = Path(r'E:\Thesis\tests\test_jupyter_notebook\outputs\baseline')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parameters aligned with backend gsplat_engine defaults/mapping.
PARAMS = {
    'mode': 'baseline',
    'max_steps': 15000,
    'eval_interval': 1000,
    'save_interval': 2500,
    'densify_from_iter': 500,
    'densify_until_iter': 10000,
    'densification_interval': 100,
    'densify_grad_threshold': 0.0002,
    'opacity_threshold': 0.005,
    'lambda_dssim': 0.2,
    'log_interval': 100,
    'splat_export_interval': 1000,
    'best_splat_interval': 100,
    'use_cuda': True,
    'run_id': 'baseline_notebook_run_001',
}

print('IMAGE_DIR  :', IMAGE_DIR)
print('COLMAP_DIR :', COLMAP_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)
print('PARAMS max_steps:', PARAMS['max_steps'])


IMAGE_DIR  : E:\Thesis\tests\b5740a69-5358-4702-a6d6-1bc57147e0a1\images_resized
COLMAP_DIR : E:\Thesis\tests\b5740a69-5358-4702-a6d6-1bc57147e0a1\outputs\sparse
OUTPUT_DIR : E:\Thesis\tests\test_jupyter_notebook\outputs\baseline
PARAMS max_steps: 15000


## 2. Import Backend Engine and Shared Helpers

In [16]:
from bimba3d_backend.worker.engines.gsplat_engine import run_training
from bimba3d_backend.worker.entrypoint import (
    _get_engine_output_dir,
    _materialize_eval_previews,
    _export_with_gsplat,
    _parse_step_from_name,
    _collect_eval_history,
    _write_json_atomic,
)

logger = logging.getLogger('notebook-gsplat')
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')


## 3. Build Independent Context (No Backend Server Needed)

In [17]:
def update_status(project_dir, status, **kwargs):
    stage = kwargs.get('stage')
    msg = kwargs.get('message')
    prog = kwargs.get('progress')
    print(f'[STATUS] {status} stage={stage} progress={prog} msg={msg}')

def write_metrics(project_dir, payload, **kwargs):
    # Mirror backend behavior by writing metrics.json in project root.
    target = Path(project_dir) / 'metrics.json'
    target.parent.mkdir(parents=True, exist_ok=True)
    with open(target, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)

context = {
    'logger': logger,
    'update_status': update_status,
    'write_metrics': write_metrics,
    'get_engine_output_dir': _get_engine_output_dir,
    'materialize_eval_previews': _materialize_eval_previews,
    'export_with_gsplat': _export_with_gsplat,
    'parse_step_from_name': _parse_step_from_name,
    'collect_eval_history': _collect_eval_history,
    'write_json_atomic': _write_json_atomic,
}


## 4. Train with Exact Backend Engine

In [ ]:
# Notebook-safe worker settings on Windows to avoid multiprocessing spawn issues.
os.environ['GSPLAT_NUM_WORKERS'] = '0'
os.environ['BIMBA3D_DOCKER_WORKER'] = '1'

result = run_training(
    image_dir=IMAGE_DIR,
    colmap_dir=COLMAP_DIR,
    output_dir=OUTPUT_DIR,
    params=PARAMS,
    resume=False,
    context=context,
)

print('Training finished. Return:', result)


2026-04-09 11:36:28,684 - INFO - Starting gsplat training (upstream simple_trainer path)...
2026-04-09 11:36:28,905 - INFO - GSPLAT logging cadence: snapshot every 100 steps (log_interval), callback every 100 steps, modified_tune_interval=100, best_splat_start_step=2000; tqdm=disabled
2026-04-09 11:36:28,980 - WARNING - MSVC env command failed (call "C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\VC\Auxiliary\Build\vcvars64.bat" && set): Command '['cmd.exe', '/d', '/s', '/c', 'call "C:\\Program Files (x86)\\Microsoft Visual Studio\\2022\\BuildTools\\VC\\Auxiliary\\Build\\vcvars64.bat" && set']' returned non-zero exit status 1.
2026-04-09 11:36:29,007 - WARNING - MSVC env command failed (call "C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\Common7\Tools\VsDevCmd.bat" -arch=x64 -host_arch=x64 -no_logo && set): Command '['cmd.exe', '/d', '/s', '/c', 'call "C:\\Program Files (x86)\\Microsoft Visual Studio\\2022\\BuildTools\\Common7\\Tools\\VsDevCmd.bat" -arc

[STATUS] processing stage=training progress=55 msg=🚀 Initializing upstream simple_trainer (GPU ⚡)...
[Parser] 69 images, taken by 1 cameras.
Scene scale: 1.5745537096987208
Model initialized. Number of GS: 49735


D:\bimba3d-re\.venv\Lib\site-packages\torchmetrics\functional\image\lpips.py:332: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_path, m

[STATUS] processing stage=training progress=60 msg=🎯 Training step 1/15000 (loss: 0.241915)


2026-04-09 11:36:45,867 - INFO - [GSPLAT SNAPSHOT] step=100/15000 progress=0.67% loss=0.167658 gs=49735 opacity_mean=-1.125115 scale_mean=-4.843207 sh_degree=n/a next_eval=1000 next_save=100 elapsed=17.0s eta=2533.0s speed=5.882 step/s tuning_applied=False strategy={'grow_grad2d': 0.0002, 'prune_opa': 0.005, 'refine_every': 100, 'reset_every': 3000} lrs={'means': 0.0002443866, 'opacities': 0.05, 'scales': 0.005, 'quats': 0.001, 'sh0': 0.0025, 'shN': 0.000125}


...

[STATUS] processing stage=training progress=70 msg=🎯 Training step 4400/15000 (loss: 0.066123)
Step:  4399 {'mem': 2.8062243461608887, 'ellipse_time': 773.8957335948944, 'num_GS': 1485628}
Step 4400: 44516 GSs duplicated, 3255 GSs split. Now having 1533399 GSs.
Step 4400: 1608 GSs pruned. Now having 1531791 GSs.


2026-04-09 11:49:47,459 - INFO - [GSPLAT SNAPSHOT] step=4500/15000 progress=30.00% loss=0.077456 gs=1531791 opacity_mean=-0.086741 scale_mean=-6.749029 sh_degree=n/a next_eval=5000 next_save=4500 elapsed=798.6s eta=1863.4s speed=5.635 step/s tuning_applied=False strategy={'grow_grad2d': 0.0002, 'prune_opa': 0.005, 'refine_every': 100, 'reset_every': 3000} lrs={'means': 6.3301e-05, 'opacities': 0.05, 'scales': 0.005, 'quats': 0.001, 'sh0': 0.0025, 'shN': 0.000125}


[STATUS] processing stage=training progress=70 msg=🎯 Training step 4500/15000 (loss: 0.077456)
Step:  4499 {'mem': 2.8628220558166504, 'ellipse_time': 795.3512864112854, 'num_GS': 1531791}
Step 4500: 41141 GSs duplicated, 2782 GSs split. Now having 1575714 GSs.
Step 4500: 1617 GSs pruned. Now having 1574097 GSs.


2026-04-09 11:50:08,834 - INFO - [GSPLAT SNAPSHOT] step=4600/15000 progress=30.67% loss=0.086025 gs=1574097 opacity_mean=-0.088294 scale_mean=-6.765213 sh_degree=n/a next_eval=5000 next_save=4600 elapsed=820.0s eta=1853.8s speed=5.610 step/s tuning_applied=False strategy={'grow_grad2d': 0.0002, 'prune_opa': 0.005, 'refine_every': 100, 'reset_every': 3000} lrs={'means': 6.13871e-05, 'opacities': 0.05, 'scales': 0.005, 'quats': 0.001, 'sh0': 0.0025, 'shN': 0.000125}


[STATUS] processing stage=training progress=70 msg=🎯 Training step 4600/15000 (loss: 0.086025)
Step:  4599 {'mem': 2.887284755706787, 'ellipse_time': 816.7260653972626, 'num_GS': 1574097}
Step 4600: 38244 GSs duplicated, 2469 GSs split. Now having 1614810 GSs.
Step 4600: 1818 GSs pruned. Now having 1612992 GSs.


## 5. Inspect Output Artifacts

In [ ]:
engine_dir = OUTPUT_DIR / 'engines' / 'gsplat'
print('Engine output dir:', engine_dir)
if engine_dir.exists():
    for p in sorted(engine_dir.glob('*')):
        if p.is_file():
            print('-', p.name, p.stat().st_size, 'bytes')
        else:
            print('-', p.name, '(dir)')
else:
    print('No engine output directory found yet.')
